<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Capítulo 2: Gerando texto com um LLM pré-treinado

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

## 2.1 Introdução a LLMs para geração de texto

- Sem código nesta seção
- Como os LLMs geram texto?
- Este é um capítulo de preparação: configurar o ambiente de programação e o LLM que usaremos ao longo do livro
- Também programamos funções de geração de texto que vamos usar e estender nos próximos capítulos

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- Fluxogramas de LLMs (e de redes neurais) são tradicionalmente lidos e desenhados de cima para baixo

## 2.2 Configurando o ambiente de programação

- Se você está lendo este livro, provavelmente já programou em Python antes
- A forma mais simples de instalar as dependências, se você já tem um ambiente Python configurado (com Python 3.10 ou mais novo), é usar o `pip`:

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- Para este capítulo, as dependências também podem ser instaladas manualmente:

In [3]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- Minha forma preferida é usar o [uv](https://docs.astral.sh/uv/), gerenciador de pacotes e projetos Python amplamente recomendado
- Para instalar o `uv`, execute a instalação correspondente ao seu sistema operacional pelo site oficial: https://docs.astral.sh/uv/getting-started/installation/
- Em seguida, clone o repositório do GitHub:

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- Se você não tem o `git` instalado, também pode baixar manualmente o repositório de código-fonte pelo site da Manning ou clicando neste link: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (descompacte depois de baixar)

- No terminal, navegue até a pasta `reasoning-from-scratch`
- O repositório inclui um arquivo `.python-version`, para que o `uv` use por padrão uma versão do Python compatível com o PyTorch
- Rode `uv run jupyter lab` para abrir o JupyterLab e abrir um notebook em branco ou o notebook deste capítulo
- Esse comando também cria um ambiente virtual local (normalmente em `.venv/`) e instala automaticamente todas as dependências do arquivo `pyproject.toml` dentro da pasta `reasoning-from-scratch`

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- Veja [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) para detalhes e opções adicionais de instalação, se precisar

## 2.3 Entendendo as necessidades e recomendações de hardware

- Se você é novo no PyTorch, recomendo ler meu tutorial [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/)
- Se você seguiu a seção anterior, já deve ter o PyTorch instalado
- Verifique manualmente se sua instalação do PyTorch suporta GPU; veja o que é suportado na sua máquina:

In [5]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


- Dependendo do capítulo, o código usará automaticamente uma GPU NVIDIA (CUDA) se disponível; caso contrário, roda em CPU (ou na GPU do Apple Silicon, se recomendado para determinada seção ou capítulo)
- Os capítulos 2 a 4 podem ser executados em um tempo razoável em CPU
- O código dos capítulos 5 a 7 será bem lento se executado em CPU, e uma GPU com suporte a CUDA é recomendada para esses capítulos (mais sobre as necessidades exatas de recursos nesses capítulos)
- Minha preferência pessoal é o [Lightning AI Studio](https://lightning.ai/), que oferece créditos de computação gratuitos após o cadastro e a verificação; como alternativa, o [Google Colab](https://colab.research.google.com/) é outra boa escolha

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- Veja [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) para recomendações de computação na nuvem, se precisar
- Mas, por ora, ainda não é necessário usar GPUs; os primeiros capítulos rodam bem em hardware sem GPU

## 2.4 Preparando textos de entrada para LLMs

- Nesta seção, aprendemos a usar um tokenizer; nós o usamos para converter (codificar) o texto de entrada em uma representação de IDs de token, como entrada para o LLM
- Também usamos o tokenizer para converter (decodificar) a saída do LLM de volta em uma representação de texto legível por humanos

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- Como mencionado antes, implementar o LLM e o tokenizer do zero está fora do escopo deste livro, que é focado em implementar métodos de raciocínio do zero sobre um LLM e um tokenizer já existentes
- Neste livro, vamos trabalhar com um LLM pré-treinado que carregaremos na próxima seção; aqui, carregamos o tokenizer que acompanha esse modelo
- Preparei um pacote Python `reasoning_from_scratch` que fornece o LLM base e o tokenizer correspondente, que programei com a ajuda do pacote da biblioteca Python [`tokenizers`](https://github.com/huggingface/tokenizers)
- O código do pacote `reasoning_from_scratch` faz parte do código suplementar deste livro e já deve estar instalado, conforme as instruções da seção 2.2

- Em seguida, baixamos os arquivos do tokenizer (este é um tokenizer para o LLM base Qwen3, mas falaremos mais sobre isso na próxima seção):

In [6]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- Agora, podemos carregar as configurações do tokenizer, do arquivo do tokenizer, para dentro do `Qwen3Tokenizer`:

In [7]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- Como ainda não carregamos o LLM em si, faremos uma ida e volta mais simples: codificamos o texto em IDs de token e depois decodificamos de volta para sua representação em string:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [8]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [9]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [10]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


- No caso do `Qwen3Tokenizer`, há cerca de 151 mil tokens únicos (tamanho do vocabulário)

- Recursos adicionais sobre tokenização:
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o), capítulo 2
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

## 2.5 Carregando modelos pré-treinados

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- Como sugerido na seção anterior, ao carregar o tokenizer, este livro usa o Qwen3 0.6B; depois de pensar muito sobre qual modelo base de pesos abertos usar, optei pelo Qwen3 porque
  - o Qwen3 é o principal modelo de pesos abertos em termos de desempenho de modelagem até o momento em que isto foi escrito
  - o Qwen3 0.6B é mais eficiente em memória que o Llama 3 1B
  - existem tanto um modelo base (no qual focamos para o desenvolvimento do modelo de raciocínio) quanto uma variante de raciocínio oficial, que podemos usar como modelo de referência
- (Note que a grafia canônica não inclui espaço em "Qwen3", enquanto inclui um em "Llama 3")
- No espírito do "do zero", estamos usando uma reimplementação do Qwen3 que escrevi em PyTorch puro, sem nenhuma dependência de biblioteca externa de LLM; essa implementação do zero é compatível com os pesos originais do modelo Qwen3
- No entanto, não vamos percorrer a implementação do código do Qwen3 neste livro, já que isso seria um livro inteiro por si só (parecido com o meu livro [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch)); em vez disso, este livro (Build A Reasoning Model From Scratch) foca em implementar métodos de raciocínio do zero sobre um modelo base (aqui, o Qwen3)
- Veja o apêndice C para o código do modelo Qwen3
- Veja o apêndice D para carregar a variante de raciocínio e modelos Qwen3 maiores
- Veja o [repositório do Qwen3 no GitHub](https://github.com/QwenLM/Qwen3) e o [relatório técnico](https://arxiv.org/abs/2505.09388) para (ainda) mais detalhes

- O modelo é propositalmente pequeno (mas ainda assim bem capaz) para rodar em hardware de consumo
- Ele roda bem em CPU, GPUs NVIDIA (`"cuda"`), GPUs Apple Silicon (`"mps"`) e GPUs Intel (`"xpu"`); mais sobre os trade-offs de desempenho adiante neste capítulo

In [11]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            # PyTorch 2.9 and 2.10 still read the legacy TF32 setting in torch.compile.
            # See https://github.com/pytorch/pytorch/issues/166387
            # and https://github.com/rasbt/reasoning-from-scratch/issues/256
            if (major, minor) >= (2, 11):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using Apple Silicon GPU (MPS)


- Recomendo rodar o código em `"cpu"` na primeira passada, então fixamos o dispositivo abaixo:

In [12]:
# Recommended: Use CPU on the first run-through
device = torch.device("cpu")

- Em seguida, baixamos o arquivo com os pesos do modelo pré-treinado, que tem aproximadamente 1,5 GB:

In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- A estrutura arquitetural do modelo Qwen3 0.6B que estamos carregando é mostrada abaixo para leitores familiarizados com arquiteturas de LLM, mas note que, para este livro, **não** é essencial nem importante entender essa arquitetura, já que não a modificamos, e sim acrescentamos técnicas de raciocínio por cima, nos capítulos seguintes

- Programei a arquitetura do modelo Qwen3 do zero para o pacote Python [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py), contido neste repositório de código; o código-fonte também é mostrado no apêndice C; mas, de novo, isso é apenas um bônus para quem tem curiosidade, e não é necessário olhar nem entender esses detalhes internos para acompanhar o restante do livro

In [14]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 Entendendo o processo sequencial de geração de texto do LLM

- Nesta seção, programamos uma função wrapper simples para podermos usar o LLM para gerar texto (vamos estender essa função com funcionalidades extras no capítulo 4)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLMs geram uma palavra por vez:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- A figura acima é uma simplificação, mostrando apenas a palavra recém-gerada; a figura abaixo dá zoom na primeira iteração:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [15]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [16]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [17]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [18]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)


In [19]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [20]:
print(tokenizer.decode([20286]))

 Large


In [21]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


&nbsp;
## 2.7 Programando uma função mínima de geração de texto


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- A função `generate_text_basic_stream` implementa esse processo sequencial de geração de texto:

In [22]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Stop if we encounter an end-of-sequence token
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # Yield each token as it's generated
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- Vamos usá-la para gerar uma resposta de 100 tokens a um prompt simples, `"Explain large language models in a single sentence."`, para ver como funciona (chegamos às partes de raciocínio nos capítulos seguintes)
- O código a seguir será lento e pode levar de 1 a 3 minutos para terminar, dependendo do seu computador (vamos acelerá-lo em seções posteriores)

In [23]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The evolution of language has

- Note que o LLM segue a instrução muito bem, mas a resposta fica sem sentido/fora do tema depois de `<|endoftext|>`, que é um token usado como delimitador entre documentos diferentes durante o treinamento
- Ao usar o LLM, queremos que ele pare de gerar depois de encontrar esse token

In [24]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


- Por conveniência, esse ID de token fica guardado como um atributo do tokenizer (eos = end of sequence):

In [25]:
print(tokenizer.eos_token_id)

151643


- Podemos usá-lo para dizer ao LLM (ou melhor, à função `generate_text_basic_stream`) quando parar de gerar texto

In [26]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # Use EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

- A resposta acima é o que você obtém rodando o código em CPU; o texto gerado pode variar ligeiramente dependendo do dispositivo

- Antes de encerrarmos esta seção e vermos como acelerar o código, vamos implementar uma função simples de benchmarking para acompanhar o desempenho computacional

In [27]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Check whether we are actually using this backend
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Synchronize if supported (important for async backends)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [28]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 1.39 sec
29 tokens/sec


&nbsp;
## 2.8 Inferência mais rápida com KV caching

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- Note que o código deste livro enfatiza a legibilidade, e um livro inteiro à parte poderia ser escrito sobre otimizações
- Aqui, olhamos para um truque de engenharia chamado "KV caching" (KV se refere às keys e values dentro do mecanismo de attention do LLM)
- Se você não conhece esses termos, não se preocupe; tudo o que você precisa saber é que há uma forma de armazenar (cachear) valores intermediários que são reutilizados a cada iteração

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- Para mais detalhes sobre a mecânica do KV caching, veja meu artigo [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)
- Abaixo está uma versão modificada da função `generate_text_basic_stream` que usa um KV cache

In [29]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # New
    model.reset_kv_cache()                           # New

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

- O uso é parecido com o anterior:

In [30]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 0.84 sec
49 tokens/sec


- Como podemos ver, é ordens de magnitude mais rápido que antes (28 tokens/sec em vez de 4 tokens/sec; rodado em uma CPU Mac Mini M4)

&nbsp;
## 2.9 Inferência mais rápida com compilação de modelo no PyTorch

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- Outra técnica para acelerar bastante a inferência do modelo (geração de texto) é usar o `torch.compile`
- O uso é simples: basta chamar `torch.compile` no modelo (veja [a documentação](https://docs.pytorch.org/docs/stable/torch.compiler_api.html) para opções adicionais)

In [31]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # This avoids retriggering model recompilations 
    # in PyTorch 2.8 and newer
    # if the model contains code like self.pos = self.pos + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer

---

**Nota sobre Windows 1**

- A compilação pode ser complicada no Windows
- O `torch.compile()` usa o Inductor, que compila kernels em JIT e precisa de um toolchain C/C++ funcional
- Para CUDA, o Inductor também depende do Triton, disponível pelo pacote da comunidade `triton-windows`
  - Se você vir `cl not found`, [instale o Visual Studio Build Tools com o "C++ workload"](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170) e rode o Python pelo prompt "x64 Native Tools"
  - Se você vir `triton not found` com CUDA, instale o `triton-windows` (por exemplo, `uv pip install "triton-windows<3.4"`).
- Para CPU, um leitor recomendou ainda seguir este [guia do PyTorch Inductor para Windows](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  - Aqui, é importante instalar o pacote de idioma inglês ao instalar o Visual Studio 2022, para evitar um erro de UTF-8
  - Note também que o código precisa ser rodado pelo "Visual Studio 2022 Developer Command Prompt", e não por um notebook
- Se essa configuração se mostrar complicada, você pode pular a compilação; **a compilação é opcional, e todos os exemplos de código funcionam bem sem ela**

**Nota sobre Windows 2**

- Leitores relataram que não há ganho de velocidade ao rodar o `torch.compile` com as configurações padrão no Windows; no entanto, rodar o `torch.compile` com o modo `"max-autotune"` resultou em um ganho de 2x: `torch.compile(model, mode="max-autotune")`

---

- A primeira iteração pode ser um pouco lenta, já que faz a compilação e a otimização iniciais; por isso, repetimos a geração de texto várias vezes
- Primeiro, vamos começar com a versão sem cache (isso pode ser um pouco lento e levar xx minutos)

In [32]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

W0213 17:02:09.090000 73246 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Warm-up run


Time: 27.15 sec
1 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 1:


Time: 0.82 sec
42 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 2:


Time: 0.82 sec
42 tokens/sec

------------------------------



- Como podemos ver acima, com 5 tokens/sec, isso é apenas marginalmente mais rápido que antes (4 tokens/sec)
- Vamos ver agora como se sai a versão com KV cache

In [33]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 45.89 sec
0 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.48 sec
84 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 2:


Time: 0.45 sec
90 tokens/sec

------------------------------



- Como podemos ver, a compilação resultou em um ganho substancial de 2x (64 tokens/sec contra 30 tokens/sec)
- Abaixo está uma tabela com resultados adicionais

| Modelo     | Modo              | Hardware             | Tokens/sec    | Memória de GPU (VRAM) |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | Normal            | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | Normal compilado  | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV cache compilado | Mac Mini M4 CPU     | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | Normal compilado  | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV cache compilado | Mac Mini M4 GPU     | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | NVIDIA H100 GPU      | 51            | 1,55 GB           |
| Qwen3Model | Normal compilado  | NVIDIA H100 GPU      | 164           | 1,81 GB           |
| Qwen3Model | KV cache          | NVIDIA H100 GPU      | 48            | 1,52 GB           |
| Qwen3Model | KV cache compilado | NVIDIA H100 GPU     | 141           | 1,81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | NVIDIA DGX Spark GPU | 74            | 1,53 GB           |
| Qwen3Model | Normal compilado  | NVIDIA DGX Spark GPU | 103           | 1,49 GB           |
| Qwen3Model | KV cache          | NVIDIA DGX Spark GPU | 68            | 1,47 GB           |
| Qwen3Model | KV cache compilado | NVIDIA DGX Spark GPU | 98            | 1,47 GB           |

- O NVIDIA DGX Spark acima usa uma GPU GB10 (Blackwell)
- Note que rodamos todos os exemplos com um único prompt (ou seja, batch size de 1); se você tem curiosidade sobre inferência em batch, veja o apêndice E

&nbsp;
## Resumo

- Sem código nesta seção